# 23 — Direction-aware curated functional enrichment and pathway interpretation

This notebook is the repository-numbered successor of the historical executed notebook:

`22_functional_enrichment_pathway_interpretation_HVG8000_r0_6.ipynb`

## Important scientific correction

A direct audit of the historical executed notebook identified a column-detection bug in the enrichment wrapper.

The upstream DE tables use these columns:

- malignant-state DE:
  - effect: `logFC_logCPM_state_vs_rest`
  - FDR: `padj_fdr_bh`
  - p-value: `pval_mannwhitney`
- condition and within-state DE:
  - effect: `log2FC_A_vs_B`
  - FDR: `padj_welch`
  - p-value: `pvalue_welch`

The historical enrichment notebook did not recognize those names. Its executed `detected_DE_columns` table therefore reported `lfc_col=None`, `q_col=None`, and `p_col=None` for all five DE inputs.

Consequently, the historical wrapper treated each loaded group as an undirected gene list and named that entire list `__up`. This mixed positively and negatively regulated genes. For example, the historical AC-like list contained all 5,208 significant state-vs-rest genes and produced strong cycling/E2F enrichment even though that list contained both directions.

That behavior is unsuitable for a directional ORA and is corrected here.

## Corrected canonical behavior

- exact upstream Notebook-21 and Notebook-22 DE outputs are required;
- direction is determined from the actual signed effect-size column;
- FDR is read from the actual upstream adjusted-p-value column;
- `up` requires `FDR <= 0.05` and `LFC >= +0.5`;
- `down` requires `FDR <= 0.05` and `LFC <= -0.5`;
- ORA remains one-sided Fisher exact enrichment against the 15,176-gene DE universe;
- BH-FDR correction, curated gene sets, interpretation summarization, program-score summaries, and plotting conventions are otherwise preserved.

## Reproducibility policy

The notebook records a `legacy_direction_detection_audit.tsv` showing exactly why the historical wrapper was direction-blind. Corrected directional gene-list QC is also written and hard-checked before ORA.

External GMT discovery is disabled by default because the historical executed run loaded zero external GMT files. This keeps the canonical analysis deterministic. External GMTs can be enabled explicitly with:

`GLIOMA_ENABLE_EXTERNAL_GMTS=1`

If enabled, the run is no longer expected to reproduce the curated-only canonical ORA exactly.

## Memory policy

This step is TSV-based and does not load any large H5AD object. The two large all-gene DE tables are scanned using only their gene column when constructing the universe.

## Windows low-commit-memory compatibility

This v2 implementation does not import SciPy or statsmodels. The one-sided Fisher exact enrichment test is evaluated directly from the hypergeometric upper tail using `math.lgamma`, and Benjamini-Hochberg correction is implemented locally. This avoids loading SciPy DLLs on Windows systems where a small paging file can prevent `_lsoda` from loading. The statistical test definition is unchanged.


In [1]:
# =========================
# 0. Imports, repository paths, and canonical inputs
# =========================
from pathlib import Path
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt


def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "metadata").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

FIG_DIR = (
    PROJECT_DIR
    / "figures"
    / "functional_enrichment_HVG8000_r0_6"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TAG = "HVG8000_r0_6"

OUT_PREFIX = (
    f"functional_enrichment_{TAG}"
)


STATE_SIG_TSV = (
    METADATA_DIR
    / f"malignant_state_pseudobulk_DE_state_vs_rest_significant_{TAG}.tsv"
)

STATE_TOP_TSV = (
    METADATA_DIR
    / f"malignant_state_pseudobulk_DE_state_vs_rest_top100_up_down_{TAG}.tsv"
)

COND_OVERALL_SIG_TSV = (
    METADATA_DIR
    / (
        f"condition_malignant_pseudobulk_DE_{TAG}_"
        "overall_malignant_pairwise_DE_significant_FDR0_05_absLFC0_5.tsv"
    )
)

COND_OVERALL_TOP_TSV = (
    METADATA_DIR
    / (
        f"condition_malignant_pseudobulk_DE_{TAG}_"
        "overall_malignant_pairwise_DE_top100_up_down.tsv"
    )
)

WITHIN_STATE_TOP_TSV = (
    METADATA_DIR
    / (
        f"condition_malignant_pseudobulk_DE_{TAG}_"
        "within_state_condition_DE_top50_up_down.tsv"
    )
)

WITHIN_STATE_SUMMARY_TSV = (
    METADATA_DIR
    / (
        f"condition_malignant_pseudobulk_DE_{TAG}_"
        "within_state_condition_DE_test_summary.tsv"
    )
)

PROGRAM_GLOBAL_TSV = (
    METADATA_DIR
    / (
        f"condition_malignant_pseudobulk_DE_{TAG}_"
        "program_score_global_kruskal.tsv"
    )
)

PROGRAM_PAIRWISE_TSV = (
    METADATA_DIR
    / (
        f"condition_malignant_pseudobulk_DE_{TAG}_"
        "program_score_pairwise_mannwhitney.tsv"
    )
)


REQUIRED_INPUTS = {
    "state_significant": STATE_SIG_TSV,
    "state_top": STATE_TOP_TSV,
    "condition_overall_significant": COND_OVERALL_SIG_TSV,
    "condition_overall_top": COND_OVERALL_TOP_TSV,
    "within_state_top": WITHIN_STATE_TOP_TSV,
    "within_state_summary": WITHIN_STATE_SUMMARY_TSV,
    "program_global": PROGRAM_GLOBAL_TSV,
    "program_pairwise": PROGRAM_PAIRWISE_TSV,
}


missing_inputs = [
    str(path)
    for path
    in REQUIRED_INPUTS.values()
    if not path.exists()
]


if missing_inputs:
    raise FileNotFoundError(
        "Notebook 23 requires the canonical Notebook-21/22 metadata outputs. "
        "Missing:\n"
        + "\n".join(
            missing_inputs
        )
    )


ENABLE_EXTERNAL_GMTS = (
    os.environ.get(
        "GLIOMA_ENABLE_EXTERNAL_GMTS",
        "0",
    )
    .strip()
    .lower()
    in {
        "1",
        "true",
        "yes",
        "on",
    }
)


print(
    "Historical notebook mapping:",
    "22_functional_enrichment_pathway_interpretation_HVG8000_r0_6.ipynb "
    "-> 23_curated_enrichment.ipynb",
)

print(
    "PROJECT_DIR:",
    PROJECT_DIR,
)

print(
    "ENABLE_EXTERNAL_GMTS:",
    ENABLE_EXTERNAL_GMTS,
)


Historical notebook mapping: 22_functional_enrichment_pathway_interpretation_HVG8000_r0_6.ipynb -> 23_curated_enrichment.ipynb
PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
ENABLE_EXTERNAL_GMTS: False


In [2]:
# =========================
# 1. I/O and statistical helpers
# =========================
def write_tsv_replace(
    df,
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        path.name
        + ".tmp"
    )

    if tmp.exists():
        tmp.unlink()

    df.to_csv(
        tmp,
        sep="\t",
        index=False,
    )

    os.replace(
        tmp,
        path,
    )

    print(
        f"[SAVED] {path} shape={df.shape}"
    )


def read_tsv_required(
    path,
):
    path = Path(
        path
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    print(
        "[READ]",
        path,
    )

    return pd.read_csv(
        path,
        sep="\t",
        low_memory=False,
    )


def first_existing_col(
    df,
    candidates,
):
    lower_map = {
        str(
            column
        ).lower(): column
        for column
        in df.columns
    }

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

        if candidate.lower() in lower_map:
            return lower_map[
                candidate.lower()
            ]

    return None


def infer_gene_col(
    df,
):
    return first_existing_col(
        df,
        [
            "gene",
            "gene_symbol",
            "symbol",
            "names",
            "index",
            "var_names",
            "features",
            "Gene",
            "GeneID",
        ],
    )


def infer_lfc_col(
    df,
):
    # Exact canonical upstream names are intentionally first.
    return first_existing_col(
        df,
        [
            "logFC_logCPM_state_vs_rest",
            "log2FC_A_vs_B",
            "log2fc",
            "log2FC",
            "logFC",
            "avg_log2FC",
            "lfc",
            "coef",
            "estimate",
        ],
    )


def infer_q_col(
    df,
):
    # Exact canonical upstream names are intentionally first.
    return first_existing_col(
        df,
        [
            "padj_fdr_bh",
            "padj_welch",
            "padj_mannwhitney",
            "q_value",
            "qvalue",
            "qval",
            "padj",
            "p_adj",
            "p_adj_bh",
            "FDR",
            "fdr",
            "adj_pval",
            "p_adjusted",
        ],
    )


def infer_p_col(
    df,
):
    # Exact canonical upstream names are intentionally first.
    return first_existing_col(
        df,
        [
            "pval_mannwhitney",
            "pvalue_welch",
            "pvalue_mannwhitney",
            "p_value",
            "pvalue",
            "pval",
            "p",
            "P.Value",
        ],
    )


def infer_group_cols(
    df,
):
    candidates = [
        "condition_malignant_state",
        "malignant_state",
        "state",
        "contrast",
        "comparison",
        "test",
        "group",
        "cluster",
        "final_annotation_refined",
        "program",
    ]

    return [
        column
        for column
        in candidates
        if column in df.columns
    ]


def standardize_de_table(
    df,
    label="de_table",
):
    if df is None or df.empty:
        return (
            pd.DataFrame(),
            {
                "label": label,
                "n_rows": 0,
                "gene_col": None,
                "lfc_col": None,
                "q_col": None,
                "p_col": None,
                "group_cols": "",
            },
        )

    d = df.copy()

    gene_col = infer_gene_col(
        d
    )

    lfc_col = infer_lfc_col(
        d
    )

    q_col = infer_q_col(
        d
    )

    p_col = infer_p_col(
        d
    )

    group_cols = infer_group_cols(
        d
    )


    if gene_col is None:
        raise ValueError(
            f"No gene column found for {label}. Columns: {list(d.columns)}"
        )


    d[
        "gene_std"
    ] = d[
        gene_col
    ].astype(str)


    d[
        "lfc_std"
    ] = (
        pd.to_numeric(
            d[
                lfc_col
            ],
            errors="coerce",
        )
        if lfc_col is not None
        else np.nan
    )


    d[
        "q_std"
    ] = (
        pd.to_numeric(
            d[
                q_col
            ],
            errors="coerce",
        )
        if q_col is not None
        else np.nan
    )


    d[
        "p_std"
    ] = (
        pd.to_numeric(
            d[
                p_col
            ],
            errors="coerce",
        )
        if p_col is not None
        else np.nan
    )


    if group_cols:
        d[
            "group_std"
        ] = (
            d[
                group_cols
            ]
            .astype(str)
            .agg(
                "__".join,
                axis=1,
            )
        )

    else:
        d[
            "group_std"
        ] = label


    d[
        "direction_std"
    ] = np.where(
        d[
            "lfc_std"
        ]
        >= 0,
        "up",
        "down",
    )


    d.loc[
        d[
            "lfc_std"
        ].isna(),
        "direction_std",
    ] = "unknown"


    info = {
        "label": label,
        "n_rows": len(
            d
        ),
        "gene_col": gene_col,
        "lfc_col": lfc_col,
        "q_col": q_col,
        "p_col": p_col,
        "group_cols": ",".join(
            group_cols
        ),
    }


    return d, info


def bh_fdr(
    pvals,
):
    pvals = np.asarray(
        pvals,
        dtype=float,
    )

    out = np.full(
        pvals.shape,
        np.nan,
        dtype=float,
    )

    valid = np.isfinite(
        pvals
    )

    if valid.sum() == 0:
        return out

    p = pvals[
        valid
    ]

    n = len(
        p
    )

    order = np.argsort(
        p
    )

    ranked = p[
        order
    ]

    adjusted = (
        ranked
        * n
        / (
            np.arange(
                n
            )
            + 1
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]

    adjusted = np.clip(
        adjusted,
        0.0,
        1.0,
    )

    restored = np.empty(
        n,
        dtype=float,
    )

    restored[
        order
    ] = adjusted

    out[
        valid
    ] = restored

    return out


def _log_comb(
    n,
    k,
):
    if (
        k < 0
        or k > n
    ):
        return -np.inf

    return (
        math.lgamma(
            n + 1
        )
        - math.lgamma(
            k + 1
        )
        - math.lgamma(
            n - k + 1
        )
    )


def fisher_exact_greater(
    a,
    b,
    c,
    d,
):
    """
    One-sided Fisher exact test for enrichment (alternative='greater').

    This reproduces the hypergeometric upper-tail probability used by
    scipy.stats.fisher_exact for a 2x2 table, without importing SciPy.
    """
    a, b, c, d = map(
        int,
        (
            a,
            b,
            c,
            d,
        ),
    )

    row1 = (
        a
        + b
    )

    row2 = (
        c
        + d
    )

    col1 = (
        a
        + c
    )

    total = (
        row1
        + row2
    )

    lower = max(
        0,
        row1
        - (
            total
            - col1
        ),
    )

    upper = min(
        row1,
        col1,
    )

    if (
        a < lower
        or a > upper
    ):
        return (
            np.nan,
            np.nan,
        )

    log_denominator = _log_comb(
        total,
        row1,
    )

    log_probabilities = [
        (
            _log_comb(
                col1,
                x,
            )
            + _log_comb(
                total
                - col1,
                row1
                - x,
            )
            - log_denominator
        )
        for x in range(
            a,
            upper + 1,
        )
    ]

    max_log = max(
        log_probabilities
    )

    p_value = (
        math.exp(
            max_log
        )
        * sum(
            math.exp(
                value
                - max_log
            )
            for value
            in log_probabilities
        )
    )

    p_value = min(
        1.0,
        float(
            p_value
        ),
    )

    denominator = (
        b
        * c
    )

    numerator = (
        a
        * d
    )

    if denominator == 0:
        if numerator > 0:
            odds_ratio = np.inf
        elif numerator == 0:
            odds_ratio = np.nan
        else:
            odds_ratio = 0.0
    else:
        odds_ratio = (
            numerator
            / denominator
        )

    return (
        float(
            odds_ratio
        ),
        p_value,
    )


def parse_gmt(
    gmt_path,
):
    gene_sets = {}

    with open(
        gmt_path,
        "r",
        encoding="utf-8",
    ) as handle:
        for line in handle:
            parts = (
                line.rstrip(
                    "\n"
                )
                .split(
                    "\t"
                )
            )

            if len(
                parts
            ) >= 3:
                name = parts[
                    0
                ]

                genes = [
                    gene.strip().upper()
                    for gene
                    in parts[
                        2:
                    ]
                    if gene.strip()
                ]

                gene_sets[
                    name
                ] = sorted(
                    set(
                        genes
                    )
                )

    return gene_sets


def collect_external_gmts():
    candidates = []

    for base in [
        PROJECT_DIR
        / "resources",
        PROJECT_DIR
        / "resources"
        / "gene_sets",
        PROJECT_DIR
        / "metadata",
        PROJECT_DIR
        / "gene_sets",
    ]:
        if base.exists():
            candidates.extend(
                sorted(
                    base.glob(
                        "*.gmt"
                    )
                )
            )

            candidates.extend(
                sorted(
                    base.glob(
                        "**/*.gmt"
                    )
                )
            )

    return sorted(
        set(
            candidates
        )
    )


def clean_gene_set_dict(
    gene_sets,
):
    output = {}

    for name, genes in gene_sets.items():
        cleaned = sorted(
            set(
                str(
                    gene
                ).upper()
                for gene
                in genes
                if (
                    pd.notna(
                        gene
                    )
                    and str(
                        gene
                    ).strip()
                )
            )
        )

        if len(
            cleaned
        ) >= 3:
            output[
                str(
                    name
                )
            ] = cleaned

    return output


def run_ora(
    gene_lists,
    gene_sets,
    universe,
    min_overlap=2,
):
    universe = set(
        str(
            gene
        ).upper()
        for gene
        in universe
        if pd.notna(
            gene
        )
    )

    results = []

    for gene_list_name, genes in gene_lists.items():
        selected = set(
            str(
                gene
            ).upper()
            for gene
            in genes
            if pd.notna(
                gene
            )
        )

        selected = (
            selected
            & universe
        )

        n_selected = len(
            selected
        )


        for set_name, set_genes in gene_sets.items():
            gs = (
                set(
                    set_genes
                )
                & universe
            )

            if (
                len(
                    gs
                )
                == 0
                or n_selected
                == 0
            ):
                continue


            overlap = (
                selected
                & gs
            )

            k = len(
                overlap
            )


            if k < min_overlap:
                continue


            a = k
            b = (
                n_selected
                - k
            )
            c = (
                len(
                    gs
                )
                - k
            )
            d = (
                len(
                    universe
                )
                - a
                - b
                - c
            )


            if min(
                a,
                b,
                c,
                d,
            ) < 0:
                continue


            odds, p_value = fisher_exact_greater(
                a,
                b,
                c,
                d,
            )


            results.append(
                {
                    "gene_list": gene_list_name,
                    "gene_set": set_name,
                    "overlap_n": a,
                    "selected_n": n_selected,
                    "gene_set_n": len(
                        gs
                    ),
                    "universe_n": len(
                        universe
                    ),
                    "odds_ratio": odds,
                    "p_value": p_value,
                    "overlap_genes": ",".join(
                        sorted(
                            overlap
                        )
                    ),
                }
            )


    result = pd.DataFrame(
        results
    )


    if len(
        result
    ) > 0:
        result[
            "q_value"
        ] = bh_fdr(
            result[
                "p_value"
            ].values
        )

        result[
            "neglog10_q"
        ] = -np.log10(
            np.clip(
                result[
                    "q_value"
                ].astype(
                    float
                ),
                1e-300,
                1.0,
            )
        )

        result = (
            result
            .sort_values(
                [
                    "q_value",
                    "p_value",
                    "odds_ratio",
                ],
                ascending=[
                    True,
                    True,
                    False,
                ],
            )
            .reset_index(
                drop=True
            )
        )


    return result


def plot_enrichment_heatmap(
    enrich_df,
    out_png,
    title,
    top_gene_sets=25,
    top_lists=30,
    value_col="neglog10_q",
):
    if (
        enrich_df is None
        or enrich_df.empty
    ):
        return None


    d = (
        enrich_df
        .copy()
        .sort_values(
            "q_value"
        )
        .head(
            500
        )
    )


    top_sets = (
        d.groupby(
            "gene_set"
        )[
            value_col
        ]
        .max()
        .sort_values(
            ascending=False
        )
        .head(
            top_gene_sets
        )
        .index
        .tolist()
    )


    top_lists_idx = (
        d.groupby(
            "gene_list"
        )[
            value_col
        ]
        .max()
        .sort_values(
            ascending=False
        )
        .head(
            top_lists
        )
        .index
        .tolist()
    )


    d = d[
        d[
            "gene_set"
        ].isin(
            top_sets
        )
        & d[
            "gene_list"
        ].isin(
            top_lists_idx
        )
    ]


    if d.empty:
        return None


    matrix = d.pivot_table(
        index="gene_set",
        columns="gene_list",
        values=value_col,
        aggfunc="max",
        fill_value=0.0,
    )


    matrix = matrix.loc[
        matrix.max(
            axis=1
        )
        .sort_values(
            ascending=False
        )
        .index
    ]


    matrix = matrix[
        matrix.max(
            axis=0
        )
        .sort_values(
            ascending=False
        )
        .index
    ]


    fig_w = max(
        8,
        0.32
        * matrix.shape[
            1
        ]
        + 4,
    )

    fig_h = max(
        6,
        0.32
        * matrix.shape[
            0
        ]
        + 2,
    )


    plt.figure(
        figsize=(
            fig_w,
            fig_h,
        )
    )

    image = plt.imshow(
        matrix.values,
        aspect="auto",
    )

    plt.colorbar(
        image,
        label="-log10(FDR)",
    )

    plt.xticks(
        range(
            matrix.shape[
                1
            ]
        ),
        matrix.columns,
        rotation=90,
        fontsize=7,
    )

    plt.yticks(
        range(
            matrix.shape[
                0
            ]
        ),
        matrix.index,
        fontsize=8,
    )

    plt.title(
        title
    )

    plt.tight_layout()


    out_png = Path(
        out_png
    )

    out_png.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.savefig(
        out_png,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()


    print(
        "[SAVED FIG]",
        out_png,
    )


    return out_png


In [3]:
# =========================
# 2. Required upstream input audit
# =========================
state_sig_raw = read_tsv_required(
    STATE_SIG_TSV
)

state_top_raw = read_tsv_required(
    STATE_TOP_TSV
)

cond_overall_sig_raw = read_tsv_required(
    COND_OVERALL_SIG_TSV
)

cond_overall_top_raw = read_tsv_required(
    COND_OVERALL_TOP_TSV
)

within_state_top_raw = read_tsv_required(
    WITHIN_STATE_TOP_TSV
)

within_state_summary_raw = read_tsv_required(
    WITHIN_STATE_SUMMARY_TSV
)

program_global = read_tsv_required(
    PROGRAM_GLOBAL_TSV
)

program_pairwise = read_tsv_required(
    PROGRAM_PAIRWISE_TSV
)


EXPECTED_INPUT_ROWS = {
    "state_significant": 20800,
    "state_top": 1000,
    "condition_overall_significant": 27004,
    "condition_overall_top": 1200,
    "within_state_top": 2400,
    "within_state_summary": 27,
    "program_global": 7,
    "program_pairwise": 42,
}


loaded_inputs = {
    "state_significant": state_sig_raw,
    "state_top": state_top_raw,
    "condition_overall_significant": cond_overall_sig_raw,
    "condition_overall_top": cond_overall_top_raw,
    "within_state_top": within_state_top_raw,
    "within_state_summary": within_state_summary_raw,
    "program_global": program_global,
    "program_pairwise": program_pairwise,
}


readiness_rows = []


for label, path in REQUIRED_INPUTS.items():
    df = loaded_inputs[
        label
    ]

    observed_rows = int(
        df.shape[
            0
        ]
    )

    expected_rows = EXPECTED_INPUT_ROWS[
        label
    ]


    if observed_rows != expected_rows:
        raise RuntimeError(
            f"Canonical upstream row-count checkpoint failed for {label}: "
            f"{observed_rows} != {expected_rows}"
        )


    readiness_rows.append(
        {
            "input": label,
            "path": str(
                path
            ),
            "exists": True,
            "n_rows": observed_rows,
            "size_mb": round(
                path.stat().st_size
                / 1024**2,
                3,
            ),
        }
    )


readiness_df = pd.DataFrame(
    readiness_rows
)


write_tsv_replace(
    readiness_df,
    METADATA_DIR
    / f"{OUT_PREFIX}_readiness.tsv",
)


if readiness_df.shape != (
    8,
    5,
):
    raise RuntimeError(
        f"Readiness checkpoint failed: {readiness_df.shape}"
    )


display(
    readiness_df
)


print(
    "PASS: exact canonical Notebook-21/22 input row counts reproduced."
)


[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_pseudobulk_DE_state_vs_rest_significant_HVG8000_r0_6.tsv
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\malignant_state_pseudobulk_DE_state_vs_rest_top100_up_down_HVG8000_r0_6.tsv
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_significant_FDR0_05_absLFC0_5.tsv
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_top100_up_down.tsv
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_within_state_condition_DE_top50_up_down.tsv
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_within_state_condition_DE_test_summary.tsv
[READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_pro

,input,path,exists,n_rows,size_mb
0,state_significant,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,20800,4.170
1,state_top,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,1000,0.210
2,condition_overall_significant,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,27004,4.511
3,condition_overall_top,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,1200,0.214
4,within_state_top,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,2400,0.483
5,within_state_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,27,0.002
6,program_global,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,7,0.001
7,program_pairwise,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,42,0.005


PASS: exact canonical Notebook-21/22 input row counts reproduced.


In [4]:
# =========================
# 3. Historical bug audit + corrected DE-column detection
# =========================
def legacy_infer_lfc_col(
    df,
):
    return first_existing_col(
        df,
        [
            "log2fc",
            "log2FC",
            "logFC",
            "avg_log2FC",
            "lfc",
            "coef",
            "estimate",
        ],
    )


def legacy_infer_q_col(
    df,
):
    return first_existing_col(
        df,
        [
            "q_value",
            "qvalue",
            "qval",
            "padj",
            "p_adj",
            "p_adj_bh",
            "FDR",
            "fdr",
            "adj_pval",
            "p_adjusted",
        ],
    )


def legacy_infer_p_col(
    df,
):
    return first_existing_col(
        df,
        [
            "p_value",
            "pvalue",
            "pval",
            "p",
            "P.Value",
        ],
    )


legacy_audit_rows = []


for label, df in [
    (
        "malignant_state_significant",
        state_sig_raw,
    ),
    (
        "malignant_state_top",
        state_top_raw,
    ),
    (
        "condition_overall_significant",
        cond_overall_sig_raw,
    ),
    (
        "condition_overall_top",
        cond_overall_top_raw,
    ),
    (
        "within_state_condition_top",
        within_state_top_raw,
    ),
]:
    legacy_audit_rows.append(
        {
            "label": label,
            "n_rows": len(
                df
            ),
            "legacy_lfc_col": legacy_infer_lfc_col(
                df
            ),
            "legacy_q_col": legacy_infer_q_col(
                df
            ),
            "legacy_p_col": legacy_infer_p_col(
                df
            ),
            "corrected_lfc_col": infer_lfc_col(
                df
            ),
            "corrected_q_col": infer_q_col(
                df
            ),
            "corrected_p_col": infer_p_col(
                df
            ),
        }
    )


legacy_direction_audit = pd.DataFrame(
    legacy_audit_rows
)


write_tsv_replace(
    legacy_direction_audit,
    METADATA_DIR
    / f"{OUT_PREFIX}_legacy_direction_detection_audit.tsv",
)


legacy_missing_all = (
    legacy_direction_audit[
        [
            "legacy_lfc_col",
            "legacy_q_col",
            "legacy_p_col",
        ]
    ]
    .isna()
    .all(
        axis=1
    )
)


if not legacy_missing_all.all():
    raise RuntimeError(
        "The historical direction-detection failure was not reproduced exactly."
    )


corrected_complete = (
    legacy_direction_audit[
        [
            "corrected_lfc_col",
            "corrected_q_col",
            "corrected_p_col",
        ]
    ]
    .notna()
    .all(
        axis=1
    )
)


if not corrected_complete.all():
    raise RuntimeError(
        "Corrected DE-column detection did not resolve every required column."
    )


display(
    legacy_direction_audit
)


state_sig, state_sig_info = standardize_de_table(
    state_sig_raw,
    "malignant_state_significant",
)

state_top, state_top_info = standardize_de_table(
    state_top_raw,
    "malignant_state_top",
)

cond_overall_sig, cond_sig_info = standardize_de_table(
    cond_overall_sig_raw,
    "condition_overall_significant",
)

cond_overall_top, cond_top_info = standardize_de_table(
    cond_overall_top_raw,
    "condition_overall_top",
)

within_state_top, within_top_info = standardize_de_table(
    within_state_top_raw,
    "within_state_condition_top",
)


detected_columns = pd.DataFrame(
    [
        state_sig_info,
        state_top_info,
        cond_sig_info,
        cond_top_info,
        within_top_info,
    ]
)


write_tsv_replace(
    detected_columns,
    METADATA_DIR
    / f"{OUT_PREFIX}_detected_DE_columns.tsv",
)


display(
    detected_columns
)


EXPECTED_CORRECTED_COLUMNS = {
    "malignant_state_significant": (
        "logFC_logCPM_state_vs_rest",
        "padj_fdr_bh",
        "pval_mannwhitney",
    ),
    "malignant_state_top": (
        "logFC_logCPM_state_vs_rest",
        "padj_fdr_bh",
        "pval_mannwhitney",
    ),
    "condition_overall_significant": (
        "log2FC_A_vs_B",
        "padj_welch",
        "pvalue_welch",
    ),
    "condition_overall_top": (
        "log2FC_A_vs_B",
        "padj_welch",
        "pvalue_welch",
    ),
    "within_state_condition_top": (
        "log2FC_A_vs_B",
        "padj_welch",
        "pvalue_welch",
    ),
}


for label, expected in EXPECTED_CORRECTED_COLUMNS.items():
    row = detected_columns.loc[
        detected_columns[
            "label"
        ]
        .astype(str)
        .eq(
            label
        )
    ]

    if len(
        row
    ) != 1:
        raise RuntimeError(
            f"Detected-column row missing/non-unique: {label}"
        )

    row = row.iloc[
        0
    ]

    observed = (
        str(
            row[
                "lfc_col"
            ]
        ),
        str(
            row[
                "q_col"
            ]
        ),
        str(
            row[
                "p_col"
            ]
        ),
    )

    if observed != expected:
        raise RuntimeError(
            f"Corrected DE-column checkpoint failed for {label}: "
            f"{observed} != {expected}"
        )


print(
    "PASS: historical direction-detection bug documented and corrected."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_legacy_direction_detection_audit.tsv shape=(5, 8)


,label,n_rows,legacy_lfc_col,legacy_q_col,legacy_p_col,corrected_lfc_col,corrected_q_col,corrected_p_col
0,malignant_state_significant,20800,None,None,None,logFC_logCPM_state_vs_rest,padj_fdr_bh,pval_mannwhitney
1,malignant_state_top,1000,None,None,None,logFC_logCPM_state_vs_rest,padj_fdr_bh,pval_mannwhitney
2,condition_overall_significant,27004,None,None,None,log2FC_A_vs_B,padj_welch,pvalue_welch
3,condition_overall_top,1200,None,None,None,log2FC_A_vs_B,padj_welch,pvalue_welch
4,within_state_condition_top,2400,None,None,None,log2FC_A_vs_B,padj_welch,pvalue_welch


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_detected_DE_columns.tsv shape=(5, 7)


,label,n_rows,gene_col,lfc_col,q_col,p_col,group_cols
0,malignant_state_significant,20800,gene,logFC_logCPM_state_vs_rest,padj_fdr_bh,pval_mannwhitney,malignant_state
1,malignant_state_top,1000,gene,logFC_logCPM_state_vs_rest,padj_fdr_bh,pval_mannwhitney,malignant_state
2,condition_overall_significant,27004,gene,log2FC_A_vs_B,padj_welch,pvalue_welch,contrast
3,condition_overall_top,1200,gene,log2FC_A_vs_B,padj_welch,pvalue_welch,contrast
4,within_state_condition_top,2400,gene,log2FC_A_vs_B,padj_welch,pvalue_welch,"condition_malignant_state,contrast"


PASS: historical direction-detection bug documented and corrected.


In [5]:
# =========================
# 4. Deterministic 15,176-gene DE universe
# =========================
UNIVERSE_FILES = [
    METADATA_DIR
    / (
        f"condition_malignant_pseudobulk_DE_{TAG}_"
        "overall_malignant_pairwise_DE_all_genes.tsv"
    ),
    METADATA_DIR
    / (
        f"condition_malignant_pseudobulk_DE_{TAG}_"
        "within_state_condition_DE_all_genes.tsv"
    ),
]


def read_gene_column_only(
    path,
):
    path = Path(
        path
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Canonical universe file missing: {path}"
        )


    header = pd.read_csv(
        path,
        sep="\t",
        nrows=0,
    )


    gene_col = infer_gene_col(
        header
    )


    if gene_col is None:
        raise RuntimeError(
            f"No gene column found in canonical universe file: {path}"
        )


    gene_df = pd.read_csv(
        path,
        sep="\t",
        usecols=[
            gene_col
        ],
        low_memory=False,
    )


    return (
        gene_df[
            gene_col
        ]
        .dropna()
        .astype(str)
        .str.upper()
    )


universe_genes = set()


for path in UNIVERSE_FILES:
    genes = read_gene_column_only(
        path
    )

    universe_genes.update(
        genes.tolist()
    )

    print(
        "[UNIVERSE READ]",
        path,
        "unique so far:",
        len(
            universe_genes
        ),
    )


if len(
    universe_genes
) != 15176:
    raise RuntimeError(
        f"Historical DE-universe checkpoint failed: "
        f"{len(universe_genes)} != 15176"
    )


universe_report = pd.DataFrame(
    [
        {
            "universe_source": "canonical_all_genes_DE_files",
            "n_universe_genes": len(
                universe_genes
            ),
            "n_universe_files": len(
                UNIVERSE_FILES
            ),
            "universe_files": ";".join(
                str(
                    path
                )
                for path
                in UNIVERSE_FILES
            ),
            "warning": "",
        }
    ]
)


write_tsv_replace(
    universe_report,
    METADATA_DIR
    / f"{OUT_PREFIX}_gene_universe_report.tsv",
)


display(
    universe_report
)


print(
    "PASS: exact 15,176-gene canonical ORA universe reproduced."
)


[UNIVERSE READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_all_genes.tsv unique so far: 15081
[UNIVERSE READ] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_within_state_condition_DE_all_genes.tsv unique so far: 15176
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_gene_universe_report.tsv shape=(1, 5)


,universe_source,n_universe_genes,n_universe_files,universe_files,warning
0,canonical_all_genes_DE_files,15176,2,G:\Repository\glioma-cnv-single-cell-atlas\met...,


PASS: exact 15,176-gene canonical ORA universe reproduced.


In [6]:
# =========================
# 5. Historical curated gene sets
# =========================
curated_gene_sets = {
    "OPC_PRONEURAL": [
        "OLIG1","OLIG2","PDGFRA","SOX10","CSPG4","PTPRZ1","BCAN","DLL3","ASCL1","PLP1","PLLP","SHISA4","NKX2-2"
    ],
    "NPC_NEURAL": [
        "SOX11","DCX","RBFOX2","ELAVL4","TUBB3","STMN2","NHLH2","SCRT2","SLC17A6","NEUROD1","NEUROD6"
    ],
    "AC_REACTIVE_ASTROCYTIC": [
        "GFAP","AQP4","ALDH1L1","CLU","ID3","ID4","SLC1A3","APOE","VIM","FAM107A","S100B","SPARCL1"
    ],
    "HYPOXIA_STRESS": [
        "NDRG1","ADM","EGLN3","P4HA1","VEGFA","SLC2A1","HILPDA","PGK1","LDHA","ENO1","BNIP3","CA9","DDIT4","ALDOA"
    ],
    "GLYCOLYSIS_METABOLIC_STRESS": [
        "SLC2A1","HK2","PFKP","ALDOA","GAPDH","PGK1","ENO1","PKM","LDHA","PDK1","TPI1"
    ],
    "MES_ECM_REACTIVE": [
        "CHI3L1","CD44","VIM","FN1","COL1A1","COL1A2","COL3A1","TNC","SERPINE1","TIMP1","TAGLN","ANXA1","LGALS3","CCL2"
    ],
    "CYCLING_G2M_MITOTIC": [
        "MKI67","TOP2A","UBE2C","CENPF","PBK","ASPM","RRM2","HMGB2","BIRC5","DLGAP5","NUSAP1","CCNB1","CCNB2","CDC20"
    ],
    "E2F_S_PHASE": [
        "MCM2","MCM3","MCM4","MCM5","MCM6","MCM7","PCNA","TYMS","RRM1","RRM2","E2F1","E2F2","CDC6","CDK2"
    ],
    "MYELOID_INFLAMMATORY": [
        "LYZ","C1QA","C1QB","C1QC","APOE","TREM2","FCN1","SPP1","AIF1","CD68","TYROBP","LST1","MS4A7","HLA-DRA","HLA-DRB1","IL1B"
    ],
    "T_NK_CYTOTOXIC": [
        "CD3D","CD3E","TRAC","TRBC1","NKG7","GNLY","GZMB","GZMH","PRF1","KLRD1","IL7R","CD8A","CD4"
    ],
    "B_PLASMA": [
        "CD79A","CD79B","MS4A1","MZB1","JCHAIN","IGHG1","IGKC","SDC1","XBP1","BANK1"
    ],
    "ENDOTHELIAL_ANGIOGENESIS": [
        "PECAM1","VWF","KDR","CLDN5","FLT1","ESAM","RAMP2","EMCN","ANGPT2","PLVAP","CDH5"
    ],
    "PERICYTE_FIBROBLAST_ECM": [
        "PDGFRB","RGS5","ACTA2","TAGLN","DCN","LUM","COL1A1","COL1A2","COL3A1","COL6A1","COL6A2","MMP2"
    ],
    "INTERFERON_RESPONSE": [
        "ISG15","IFIT1","IFIT2","IFIT3","MX1","MX2","OAS1","OAS2","STAT1","IRF7","IFI6","IFI44L","CXCL10"
    ],
    "TNFA_NFKB_INFLAMMATION": [
        "NFKBIA","TNFAIP3","RELB","IL6","IL1B","CXCL8","CCL2","CCL3","CCL4","ICAM1","JUNB","FOS"
    ],
    "DNA_REPAIR_DAMAGE_RESPONSE": [
        "BRCA1","BRCA2","RAD51","MRE11","ATM","ATR","CHEK1","CHEK2","TP53","GADD45A","DDB2","XPC"
    ],
}


gene_sets = clean_gene_set_dict(
    curated_gene_sets
)


external_loaded = []


if ENABLE_EXTERNAL_GMTS:
    external_gmts = collect_external_gmts()

    for gmt in external_gmts:
        try:
            parsed = clean_gene_set_dict(
                parse_gmt(
                    gmt
                )
            )

            for name, genes in parsed.items():
                gene_sets[
                    f"{gmt.stem}::{name}"
                ] = genes

            external_loaded.append(
                {
                    "gmt_file": str(
                        gmt
                    ),
                    "n_gene_sets": len(
                        parsed
                    ),
                    "status": "loaded",
                }
            )

        except Exception as error:
            external_loaded.append(
                {
                    "gmt_file": str(
                        gmt
                    ),
                    "n_gene_sets": 0,
                    "status": repr(
                        error
                    ),
                }
            )


if not ENABLE_EXTERNAL_GMTS:
    if len(
        gene_sets
    ) != 16:
        raise RuntimeError(
            f"Historical curated-gene-set count failed: {len(gene_sets)} != 16"
        )


gene_set_inventory = pd.DataFrame(
    [
        {
            "gene_set": name,
            "n_genes_raw": len(
                genes
            ),
            "n_genes_in_universe": len(
                set(
                    genes
                )
                & set(
                    universe_genes
                )
            ),
            "source": (
                "external_gmt"
                if "::"
                in name
                else "curated_internal"
            ),
        }
        for name, genes
        in gene_sets.items()
    ]
).sort_values(
    [
        "source",
        "gene_set",
    ]
).reset_index(
    drop=True
)


external_loaded_df = pd.DataFrame(
    external_loaded,
    columns=[
        "gmt_file",
        "n_gene_sets",
        "status",
    ],
)


write_tsv_replace(
    gene_set_inventory,
    METADATA_DIR
    / f"{OUT_PREFIX}_gene_set_inventory.tsv",
)


write_tsv_replace(
    external_loaded_df,
    METADATA_DIR
    / f"{OUT_PREFIX}_external_gmt_files_loaded.tsv",
)


if not ENABLE_EXTERNAL_GMTS:
    if gene_set_inventory.shape != (
        16,
        4,
    ):
        raise RuntimeError(
            f"Historical curated inventory checkpoint failed: "
            f"{gene_set_inventory.shape}"
        )


display(
    gene_set_inventory
)


display(
    external_loaded_df
)


print(
    "PASS: curated gene-set inventory prepared."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_gene_set_inventory.tsv shape=(16, 4)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_external_gmt_files_loaded.tsv shape=(0, 3)


,gene_set,n_genes_raw,n_genes_in_universe,source
0,AC_REACTIVE_ASTROCYTIC,12,12,curated_internal
1,B_PLASMA,10,8,curated_internal
2,CYCLING_G2M_MITOTIC,14,14,curated_internal
3,DNA_REPAIR_DAMAGE_RESPONSE,12,11,curated_internal
4,E2F_S_PHASE,14,14,curated_internal
5,ENDOTHELIAL_ANGIOGENESIS,11,11,curated_internal
6,GLYCOLYSIS_METABOLIC_STRESS,11,11,curated_internal
7,HYPOXIA_STRESS,14,14,curated_internal
8,INTERFERON_RESPONSE,13,13,curated_internal
9,MES_ECM_REACTIVE,14,14,curated_internal


,gmt_file,n_gene_sets,status


PASS: curated gene-set inventory prepared.


In [7]:
# =========================
# 6. Corrected directional gene lists
# =========================
Q_CUT = 0.05
ABS_LFC_CUT = 0.5


def make_directional_gene_lists(
    df,
    label_prefix,
    q_cut=Q_CUT,
    abs_lfc_cut=ABS_LFC_CUT,
):
    if df is None or df.empty:
        return {}, pd.DataFrame()


    required = [
        "gene_std",
        "lfc_std",
        "q_std",
        "group_std",
    ]


    missing = [
        column
        for column
        in required
        if column
        not in df.columns
    ]


    if missing:
        raise RuntimeError(
            f"Standardized DE table lacks columns: {missing}"
        )


    if not df[
        "lfc_std"
    ].notna().any():
        raise RuntimeError(
            f"{label_prefix}: no usable effect sizes were detected."
        )


    if not df[
        "q_std"
    ].notna().any():
        raise RuntimeError(
            f"{label_prefix}: no usable adjusted p-values were detected."
        )


    gene_lists = {}
    qc_rows = []


    for group, sub in df.groupby(
        "group_std",
        dropna=False,
    ):
        sub = sub.copy()


        up_rows = sub[
            (
                sub[
                    "q_std"
                ]
                <= q_cut
            )
            & (
                sub[
                    "lfc_std"
                ]
                >= abs_lfc_cut
            )
        ].copy()


        down_rows = sub[
            (
                sub[
                    "q_std"
                ]
                <= q_cut
            )
            & (
                sub[
                    "lfc_std"
                ]
                <= -abs_lfc_cut
            )
        ].copy()


        for direction, selected_rows in [
            (
                "up",
                up_rows,
            ),
            (
                "down",
                down_rows,
            ),
        ]:
            genes = (
                selected_rows[
                    "gene_std"
                ]
                .dropna()
                .astype(str)
                .drop_duplicates()
                .tolist()
            )


            if len(
                genes
            ) >= 3:
                list_name = (
                    f"{label_prefix}__{group}__{direction}"
                )

                gene_lists[
                    list_name
                ] = genes


                qc_rows.append(
                    {
                        "analysis": label_prefix,
                        "gene_list": list_name,
                        "group": str(
                            group
                        ),
                        "direction": direction,
                        "n_genes": len(
                            genes
                        ),
                        "min_lfc": float(
                            selected_rows[
                                "lfc_std"
                            ].min()
                        ),
                        "max_lfc": float(
                            selected_rows[
                                "lfc_std"
                            ].max()
                        ),
                        "max_q": float(
                            selected_rows[
                                "q_std"
                            ].max()
                        ),
                    }
                )


    return (
        gene_lists,
        pd.DataFrame(
            qc_rows
        ),
    )


# Prefer significant tables, exactly as the historical wrapper did.
state_source = (
    state_sig
    if len(
        state_sig
    ) > 0
    else state_top
)


condition_source = (
    cond_overall_sig
    if len(
        cond_overall_sig
    ) > 0
    else cond_overall_top
)


state_lists, state_list_qc = make_directional_gene_lists(
    state_source,
    "state_marker",
)


condition_overall_lists, condition_list_qc = make_directional_gene_lists(
    condition_source,
    "condition_overall",
)


within_state_lists, within_list_qc = make_directional_gene_lists(
    within_state_top,
    "within_state_condition",
)


direction_qc = pd.concat(
    [
        state_list_qc,
        condition_list_qc,
        within_list_qc,
    ],
    ignore_index=True,
)


if direction_qc.empty:
    raise RuntimeError(
        "No corrected directional gene lists were generated."
    )


# Hard semantic checks: list names and selected effect signs must agree.
up_qc = direction_qc[
    direction_qc[
        "direction"
    ]
    .eq(
        "up"
    )
]


down_qc = direction_qc[
    direction_qc[
        "direction"
    ]
    .eq(
        "down"
    )
]


if not (
    up_qc[
        "min_lfc"
    ]
    >= ABS_LFC_CUT
).all():
    raise RuntimeError(
        "At least one corrected UP list contains an effect below +0.5."
    )


if not (
    down_qc[
        "max_lfc"
    ]
    <= -ABS_LFC_CUT
).all():
    raise RuntimeError(
        "At least one corrected DOWN list contains an effect above -0.5."
    )


if not (
    direction_qc[
        "max_q"
    ]
    <= Q_CUT
).all():
    raise RuntimeError(
        "At least one corrected list contains a gene with FDR > 0.05."
    )


write_tsv_replace(
    direction_qc,
    METADATA_DIR
    / f"{OUT_PREFIX}_directional_gene_list_QC.tsv",
)


gene_list_inventory = (
    direction_qc[
        [
            "analysis",
            "gene_list",
            "n_genes",
        ]
    ]
    .copy()
    .sort_values(
        [
            "analysis",
            "gene_list",
        ]
    )
    .reset_index(
        drop=True
    )
)


write_tsv_replace(
    gene_list_inventory,
    METADATA_DIR
    / f"{OUT_PREFIX}_gene_list_inventory.tsv",
)


display(
    direction_qc
)


print(
    "Corrected list counts:",
    direction_qc.groupby(
        [
            "analysis",
            "direction",
        ]
    )
    .size()
    .to_dict(),
)


print(
    "PASS: every canonical ORA gene list is explicitly direction-aware."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_directional_gene_list_QC.tsv shape=(70, 8)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_gene_list_inventory.tsv shape=(70, 3)


,analysis,gene_list,group,direction,n_genes,min_lfc,max_lfc,max_q
0,state_marker,state_marker__AC_like_reactive_malignant__up,AC_like_reactive_malignant,up,1201,0.500891,2.973679,4.977184e-02
1,state_marker,state_marker__AC_like_reactive_malignant__down,AC_like_reactive_malignant,down,853,-2.003457,-0.500238,3.987355e-02
2,state_marker,state_marker__NPC_neuronal_like_malignant__up,NPC_neuronal_like_malignant,up,1489,0.500075,4.687896,4.840770e-02
3,state_marker,state_marker__NPC_neuronal_like_malignant__down,NPC_neuronal_like_malignant,down,2261,-2.993475,-0.500104,4.986380e-02
4,state_marker,state_marker__OPC_NPC_like_malignant__up,OPC_NPC_like_malignant,up,268,0.500139,1.751733,4.957288e-02
...,...,...,...,...,...,...,...,...
65,within_state_condition,within_state_condition__hypoxia_stress_maligna...,hypoxia_stress_malignant__GBM_vs_ndGBM,down,50,-3.673176,-0.934371,1.250292e-06
66,within_state_condition,within_state_condition__hypoxia_stress_maligna...,hypoxia_stress_malignant__GBM_vs_rGBM,up,50,0.603446,5.292447,1.061193e-05
67,within_state_condition,within_state_condition__hypoxia_stress_maligna...,hypoxia_stress_malignant__GBM_vs_rGBM,down,50,-3.468383,-1.013810,1.469482e-08
68,within_state_condition,within_state_condition__hypoxia_stress_maligna...,hypoxia_stress_malignant__ndGBM_vs_rGBM,up,50,0.576216,4.007755,1.434066e-02


Corrected list counts: {('condition_overall', 'down'): 6, ('condition_overall', 'up'): 6, ('state_marker', 'down'): 5, ('state_marker', 'up'): 5, ('within_state_condition', 'down'): 24, ('within_state_condition', 'up'): 24}
PASS: every canonical ORA gene list is explicitly direction-aware.


In [8]:
# =========================
# 7. Corrected curated ORA
# =========================
state_enrich = run_ora(
    state_lists,
    gene_sets,
    universe_genes,
    min_overlap=2,
)


condition_overall_enrich = run_ora(
    condition_overall_lists,
    gene_sets,
    universe_genes,
    min_overlap=2,
)


within_state_enrich = run_ora(
    within_state_lists,
    gene_sets,
    universe_genes,
    min_overlap=2,
)


write_tsv_replace(
    state_enrich,
    METADATA_DIR
    / f"{OUT_PREFIX}_malignant_state_marker_ORA.tsv",
)


write_tsv_replace(
    condition_overall_enrich,
    METADATA_DIR
    / f"{OUT_PREFIX}_condition_overall_malignant_ORA.tsv",
)


write_tsv_replace(
    within_state_enrich,
    METADATA_DIR
    / f"{OUT_PREFIX}_within_state_condition_ORA.tsv",
)


for label, result in [
    (
        "state_marker",
        state_enrich,
    ),
    (
        "condition_overall",
        condition_overall_enrich,
    ),
    (
        "within_state_condition",
        within_state_enrich,
    ),
]:
    if not result.empty:
        if not (
            result[
                "universe_n"
            ]
            == 15176
        ).all():
            raise RuntimeError(
                f"{label}: ORA universe changed unexpectedly."
            )

        if (
            result[
                "selected_n"
            ]
            <= 0
        ).any():
            raise RuntimeError(
                f"{label}: ORA contains an empty selected list."
            )


print(
    "ORA shapes:",
    {
        "state_marker": state_enrich.shape,
        "condition_overall": condition_overall_enrich.shape,
        "within_state_condition": within_state_enrich.shape,
    },
)


display(
    state_enrich.head(
        40
    )
)


display(
    condition_overall_enrich.head(
        40
    )
)


display(
    within_state_enrich.head(
        40
    )
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_malignant_state_marker_ORA.tsv shape=(54, 11)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_condition_overall_malignant_ORA.tsv shape=(73, 11)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_within_state_condition_ORA.tsv shape=(0, 0)
ORA shapes: {'state_marker': (54, 11), 'condition_overall': (73, 11), 'within_state_condition': (0, 0)}


,gene_list,gene_set,overlap_n,selected_n,gene_set_n,universe_n,odds_ratio,p_value,overlap_genes,q_value,neglog10_q
0,state_marker__OPC_NPC_like_malignant__down,CYCLING_G2M_MITOTIC,12,241,14,15176,391.257642,1.730536e-20,"ASPM,CCNB1,CCNB2,CDC20,CENPF,DLGAP5,HMGB2,NUSA...",9.344894e-19,18.029426
1,state_marker__hypoxia_stress_malignant__up,HYPOXIA_STRESS,14,681,14,15176,inf,1.180433e-19,"ADM,ALDOA,BNIP3,CA9,DDIT4,EGLN3,ENO1,HILPDA,LD...",3.187168e-18,17.496595
2,state_marker__AC_like_reactive_malignant__down,CYCLING_G2M_MITOTIC,14,853,14,15176,inf,2.838645e-18,"ASPM,BIRC5,CCNB1,CCNB2,CDC20,CENPF,DLGAP5,HMGB...",5.109560e-17,16.291616
3,state_marker__cycling_malignant_candidate__up,CYCLING_G2M_MITOTIC,14,894,14,15176,inf,5.504233e-18,"ASPM,BIRC5,CCNB1,CCNB2,CDC20,CENPF,DLGAP5,HMGB...",5.944572e-17,16.225879
4,state_marker__cycling_malignant_candidate__up,E2F_S_PHASE,14,894,14,15176,inf,5.504233e-18,"CDC6,CDK2,E2F1,E2F2,MCM2,MCM3,MCM4,MCM5,MCM6,M...",5.944572e-17,16.225879
5,state_marker__hypoxia_stress_malignant__up,GLYCOLYSIS_METABOLIC_STRESS,11,681,11,15176,inf,1.374691e-15,"ALDOA,ENO1,GAPDH,HK2,LDHA,PDK1,PFKP,PGK1,PKM,S...",1.237222e-14,13.907552
6,state_marker__hypoxia_stress_malignant__up,MES_ECM_REACTIVE,12,681,14,15176,129.982063,5.085922e-15,"ANXA1,CCL2,CD44,CHI3L1,COL1A1,FN1,LGALS3,SERPI...",3.923425e-14,13.406335
7,state_marker__AC_like_reactive_malignant__up,AC_REACTIVE_ASTROCYTIC,12,1201,12,15176,inf,5.735544e-14,"ALDH1L1,APOE,AQP4,CLU,FAM107A,GFAP,ID3,ID4,S10...",3.871492e-13,12.412122
8,state_marker__AC_like_reactive_malignant__down,E2F_S_PHASE,12,853,14,15176,102.171225,7.568971e-14,"CDC6,E2F1,E2F2,MCM2,MCM4,MCM5,MCM6,MCM7,PCNA,R...",4.541383e-13,12.342812
9,state_marker__OPC_NPC_like_malignant__up,OPC_PRONEURAL,9,268,13,15176,129.474903,9.823521e-14,"ASCL1,BCAN,CSPG4,DLL3,OLIG1,OLIG2,PLLP,PTPRZ1,...",5.304701e-13,12.275339


,gene_list,gene_set,overlap_n,selected_n,gene_set_n,universe_n,odds_ratio,p_value,overlap_genes,q_value,neglog10_q
0,condition_overall__LGG_vs_ndGBM__down,MES_ECM_REACTIVE,14,3011,14,15176,inf,1.429578e-10,"ANXA1,CCL2,CD44,CHI3L1,COL1A1,COL1A2,COL3A1,FN...",1.043592e-08,7.981469
1,condition_overall__GBM_vs_LGG__up,MES_ECM_REACTIVE,14,4130,14,15176,inf,1.202607e-08,"ANXA1,CCL2,CD44,CHI3L1,COL1A1,COL1A2,COL3A1,FN...",4.389514e-07,6.357584
2,condition_overall__LGG_vs_rGBM__down,MES_ECM_REACTIVE,13,3575,14,15176,42.335766,7.405422e-08,"ANXA1,CCL2,CD44,CHI3L1,COL1A1,COL1A2,FN1,LGALS...",1.801986e-06,5.744249
3,condition_overall__LGG_vs_ndGBM__down,PERICYTE_FIBROBLAST_ECM,11,3011,12,15176,44.601333,1.815538e-07,"ACTA2,COL1A1,COL1A2,COL3A1,COL6A1,COL6A2,DCN,L...",3.313358e-06,5.479732
4,condition_overall__LGG_vs_ndGBM__down,HYPOXIA_STRESS,11,3011,14,15176,14.864667,3.694606e-06,"ADM,ALDOA,CA9,ENO1,HILPDA,LDHA,NDRG1,P4HA1,PGK...",5.394125e-05,4.268079
5,condition_overall__LGG_vs_rGBM__down,T_NK_CYTOTOXIC,10,3575,11,15176,32.538569,4.508034e-06,"CD3D,CD3E,CD4,GNLY,GZMB,IL7R,KLRD1,NKG7,TRAC,T...",5.484774e-05,4.260841
6,condition_overall__LGG_vs_ndGBM__down,GLYCOLYSIS_METABOLIC_STRESS,9,3011,11,15176,18.232345,1.753785e-05,"ALDOA,ENO1,GAPDH,HK2,LDHA,PGK1,PKM,SLC2A1,TPI1",1.600329e-04,3.795791
7,condition_overall__LGG_vs_ndGBM__down,T_NK_CYTOTOXIC,9,3011,11,15176,18.232345,1.753785e-05,"CD3D,CD3E,CD8A,GNLY,IL7R,KLRD1,NKG7,TRAC,TRBC1",1.600329e-04,3.795791
8,condition_overall__LGG_vs_rGBM__down,HYPOXIA_STRESS,11,3575,14,15176,11.932099,2.157058e-05,"ADM,ALDOA,CA9,ENO1,HILPDA,LDHA,NDRG1,P4HA1,PGK...",1.749614e-04,3.757058
9,condition_overall__LGG_vs_ndGBM__down,MYELOID_INFLAMMATORY,11,3011,16,15176,8.917333,2.978511e-05,"C1QA,C1QB,C1QC,CD68,HLA-DRA,HLA-DRB1,LST1,LYZ,...",2.174313e-04,3.662678


""


In [9]:
# =========================
# 8. Biological interpretation summary
# =========================
def summarize_top_enrichment(
    enrich_df,
    analysis_label,
    top_n_per_list=5,
    q_cut=0.10,
):
    if (
        enrich_df is None
        or enrich_df.empty
    ):
        return pd.DataFrame()


    rows = []


    for gene_list, sub in enrich_df.groupby(
        "gene_list"
    ):
        sub = (
            sub
            .sort_values(
                [
                    "q_value",
                    "p_value",
                    "odds_ratio",
                ],
                ascending=[
                    True,
                    True,
                    False,
                ],
            )
        )


        sub = (
            sub[
                sub[
                    "q_value"
                ]
                <= q_cut
            ]
            .head(
                top_n_per_list
            )
        )


        for _, row in sub.iterrows():
            rows.append(
                {
                    "analysis": analysis_label,
                    "gene_list": gene_list,
                    "gene_set": row[
                        "gene_set"
                    ],
                    "q_value": row[
                        "q_value"
                    ],
                    "odds_ratio": row[
                        "odds_ratio"
                    ],
                    "overlap_n": row[
                        "overlap_n"
                    ],
                    "selected_n": row[
                        "selected_n"
                    ],
                    "overlap_genes": row[
                        "overlap_genes"
                    ],
                }
            )


    return pd.DataFrame(
        rows
    )


interpretation_parts = [
    summarize_top_enrichment(
        state_enrich,
        "state_marker",
        top_n_per_list=5,
        q_cut=0.10,
    ),
    summarize_top_enrichment(
        condition_overall_enrich,
        "condition_overall",
        top_n_per_list=5,
        q_cut=0.10,
    ),
    summarize_top_enrichment(
        within_state_enrich,
        "within_state_condition",
        top_n_per_list=3,
        q_cut=0.10,
    ),
]


interpretation_parts = [
    part
    for part
    in interpretation_parts
    if not part.empty
]


interpretation_summary = (
    pd.concat(
        interpretation_parts,
        ignore_index=True,
    )
    if interpretation_parts
    else pd.DataFrame(
        columns=[
            "analysis",
            "gene_list",
            "gene_set",
            "q_value",
            "odds_ratio",
            "overlap_n",
            "selected_n",
            "overlap_genes",
        ]
    )
)


write_tsv_replace(
    interpretation_summary,
    METADATA_DIR
    / f"{OUT_PREFIX}_biological_interpretation_summary.tsv",
)


display(
    interpretation_summary.head(
        100
    )
)


print(
    "Interpretation summary shape:",
    interpretation_summary.shape,
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_biological_interpretation_summary.tsv shape=(70, 8)


,analysis,gene_list,gene_set,q_value,odds_ratio,overlap_n,selected_n,overlap_genes
0,state_marker,state_marker__AC_like_reactive_malignant__down,CYCLING_G2M_MITOTIC,5.109560e-17,inf,14,853,"ASPM,BIRC5,CCNB1,CCNB2,CDC20,CENPF,DLGAP5,HMGB..."
1,state_marker,state_marker__AC_like_reactive_malignant__down,E2F_S_PHASE,4.541383e-13,102.171225,12,853,"CDC6,E2F1,E2F2,MCM2,MCM4,MCM5,MCM6,MCM7,PCNA,R..."
2,state_marker,state_marker__AC_like_reactive_malignant__down,OPC_PRONEURAL,2.740990e-07,27.111006,8,853,"ASCL1,CSPG4,DLL3,NKX2-2,OLIG1,OLIG2,PDGFRA,PLLP"
3,state_marker,state_marker__AC_like_reactive_malignant__down,NPC_NEURAL,1.203098e-06,29.619681,7,853,"DCX,ELAVL4,NEUROD1,RBFOX2,SCRT2,SOX11,STMN2"
4,state_marker,state_marker__AC_like_reactive_malignant__down,DNA_REPAIR_DAMAGE_RESPONSE,2.672946e-02,6.315441,3,853,"BRCA2,CHEK1,RAD51"
...,...,...,...,...,...,...,...,...
65,condition_overall,condition_overall__LGG_vs_rGBM__up,OPC_PRONEURAL,6.267816e-03,6.241276,7,2395,"ASCL1,BCAN,DLL3,OLIG1,OLIG2,PDGFRA,PLP1"
66,condition_overall,condition_overall__LGG_vs_rGBM__up,NPC_NEURAL,4.049913e-02,4.454324,5,2395,"DCX,ELAVL4,RBFOX2,SOX11,TUBB3"
67,condition_overall,condition_overall__ndGBM_vs_rGBM__down,E2F_S_PHASE,3.061071e-04,17.404643,5,474,"E2F1,MCM2,MCM6,MCM7,RRM1"
68,condition_overall,condition_overall__ndGBM_vs_rGBM__up,ENDOTHELIAL_ANGIOGENESIS,2.204007e-02,17.148912,2,196,"ANGPT2,CLDN5"


Interpretation summary shape: (70, 8)


In [10]:
# =========================
# 9. Program-score summaries from canonical Notebook 22
# =========================
program_global_out = (
    program_global.copy()
)

program_pairwise_out = (
    program_pairwise.copy()
)


if program_global_out.shape != (
    7,
    9,
):
    raise RuntimeError(
        f"Canonical program-global shape failed: {program_global_out.shape}"
    )


if program_pairwise_out.shape != (
    42,
    10,
):
    raise RuntimeError(
        f"Canonical program-pairwise shape failed: {program_pairwise_out.shape}"
    )


write_tsv_replace(
    program_global_out,
    METADATA_DIR
    / f"{OUT_PREFIX}_program_score_global_kruskal_summary.tsv",
)


write_tsv_replace(
    program_pairwise_out,
    METADATA_DIR
    / f"{OUT_PREFIX}_program_score_pairwise_summary.tsv",
)


display(
    program_global_out
)


display(
    program_pairwise_out.head(
        50
    )
)


print(
    "PASS: Notebook-22 program-score summaries propagated unchanged."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_program_score_global_kruskal_summary.tsv shape=(7, 9)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_program_score_pairwise_summary.tsv shape=(42, 10)


,program,score_col,test,pvalue,median_GBM,median_LGG,median_ndGBM,median_rGBM,padj
0,MES_like,MES_like_score,Kruskal-Wallis,0.006858,0.080356,-2.272040,-0.279584,0.121846,0.037022
1,hypoxia_stress,hypoxia_stress_score,Kruskal-Wallis,0.020526,0.139258,-1.257060,0.014455,0.063136,0.037022
2,OPC_proneural,OPC_proneural_score,Kruskal-Wallis,0.022535,0.022480,1.117839,0.054328,0.170847,0.037022
3,cycling,cycling_score,Kruskal-Wallis,0.025211,0.203176,-0.427764,0.318749,0.424719,0.037022
4,NPC_neural,NPC_neural_score,Kruskal-Wallis,0.026444,-0.099047,0.879344,-0.147974,-0.264109,0.037022
5,immune_myeloid_review,immune_myeloid_review_score,Kruskal-Wallis,0.036276,0.383883,-0.875764,-0.203131,-0.055433,0.042322
6,AC_reactive,AC_reactive_score,Kruskal-Wallis,0.098234,-0.042457,-1.014897,0.253113,0.142264,0.098234


,program,condition_A,condition_B,n_A,n_B,median_A,median_B,delta_median_A_minus_B,pvalue_mannwhitney,padj_mannwhitney
0,MES_like,GBM,LGG,39,4,0.080356,-2.272040,2.352396,0.000016,0.000681
1,MES_like,LGG,ndGBM,4,18,-2.272040,-0.279584,-1.992457,0.000273,0.004594
2,MES_like,LGG,rGBM,4,16,-2.272040,0.121846,-2.393886,0.000413,0.004594
3,hypoxia_stress,GBM,LGG,39,4,0.139258,-1.257060,1.396317,0.000438,0.004594
4,OPC_proneural,GBM,LGG,39,4,0.022480,1.117839,-1.095359,0.000616,0.005173
5,NPC_neural,LGG,rGBM,4,16,0.879344,-0.264109,1.143453,0.000826,0.005779
6,OPC_proneural,LGG,rGBM,4,16,1.117839,0.170847,0.946992,0.001651,0.008669
7,hypoxia_stress,LGG,rGBM,4,16,-1.257060,0.063136,-1.320196,0.001651,0.008669
8,hypoxia_stress,LGG,ndGBM,4,18,-1.257060,0.014455,-1.271515,0.001914,0.008931
9,OPC_proneural,LGG,ndGBM,4,18,1.117839,0.054328,1.063511,0.003281,0.013780


PASS: Notebook-22 program-score summaries propagated unchanged.


In [11]:
# =========================
# 10. Corrected enrichment figures
# =========================
figure_records = []


path = plot_enrichment_heatmap(
    state_enrich,
    FIG_DIR
    / f"{OUT_PREFIX}_state_marker_ORA_heatmap.png",
    "Malignant-state directional marker enrichment",
    top_gene_sets=25,
    top_lists=20,
)


if path is not None:
    figure_records.append(
        {
            "figure": "state_marker_ORA_heatmap",
            "path": str(
                path
            ),
        }
    )


path = plot_enrichment_heatmap(
    condition_overall_enrich,
    FIG_DIR
    / f"{OUT_PREFIX}_condition_overall_ORA_heatmap.png",
    "Condition-specific directional malignant pseudobulk enrichment",
    top_gene_sets=25,
    top_lists=30,
)


if path is not None:
    figure_records.append(
        {
            "figure": "condition_overall_ORA_heatmap",
            "path": str(
                path
            ),
        }
    )


path = plot_enrichment_heatmap(
    within_state_enrich,
    FIG_DIR
    / f"{OUT_PREFIX}_within_state_condition_ORA_heatmap.png",
    "Within-state condition-specific directional enrichment",
    top_gene_sets=25,
    top_lists=35,
)


if path is not None:
    figure_records.append(
        {
            "figure": "within_state_condition_ORA_heatmap",
            "path": str(
                path
            ),
        }
    )


prog_col = first_existing_col(
    program_global,
    [
        "program",
        "gene_program",
        "score",
        "module",
    ],
)


q_col = infer_q_col(
    program_global
)


p_col = infer_p_col(
    program_global
)


if (
    prog_col is not None
    and (
        q_col is not None
        or p_col is not None
    )
):
    d = (
        program_global
        .copy()
    )

    val_col = (
        q_col
        if q_col is not None
        else p_col
    )

    d[
        "val"
    ] = pd.to_numeric(
        d[
            val_col
        ],
        errors="coerce",
    )

    d = (
        d.dropna(
            subset=[
                "val"
            ]
        )
        .sort_values(
            "val"
        )
    )

    d[
        "neglog10"
    ] = -np.log10(
        np.clip(
            d[
                "val"
            ].values,
            1e-300,
            1.0,
        )
    )


    plt.figure(
        figsize=(
            8,
            max(
                4,
                0.35
                * len(
                    d
                )
                + 1,
            ),
        )
    )


    plt.barh(
        range(
            len(
                d
            )
        ),
        d[
            "neglog10"
        ].values,
    )


    plt.yticks(
        range(
            len(
                d
            )
        ),
        d[
            prog_col
        ]
        .astype(str)
        .tolist(),
    )


    plt.xlabel(
        f"-log10({val_col})"
    )


    plt.title(
        "Condition-associated malignant program score differences"
    )


    plt.gca().invert_yaxis()


    output_path = (
        FIG_DIR
        / f"{OUT_PREFIX}_program_score_global_barplot.png"
    )


    plt.tight_layout()


    plt.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )


    plt.close()


    figure_records.append(
        {
            "figure": "program_score_global_barplot",
            "path": str(
                output_path
            ),
        }
    )


figure_manifest = pd.DataFrame(
    figure_records
)


write_tsv_replace(
    figure_manifest,
    METADATA_DIR
    / f"{OUT_PREFIX}_figure_manifest.tsv",
)


if not (
    figure_manifest[
        "figure"
    ]
    .astype(str)
    .isin(
        [
            "state_marker_ORA_heatmap",
            "condition_overall_ORA_heatmap",
            "within_state_condition_ORA_heatmap",
            "program_score_global_barplot",
        ]
    )
    .all()
):
    raise RuntimeError(
        "Unexpected figure type in canonical enrichment manifest."
    )


display(
    figure_manifest
)


print(
    "Figure manifest shape:",
    figure_manifest.shape,
)


[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\functional_enrichment_HVG8000_r0_6\functional_enrichment_HVG8000_r0_6_state_marker_ORA_heatmap.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\functional_enrichment_HVG8000_r0_6\functional_enrichment_HVG8000_r0_6_condition_overall_ORA_heatmap.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_figure_manifest.tsv shape=(3, 2)


,figure,path
0,state_marker_ORA_heatmap,G:\Repository\glioma-cnv-single-cell-atlas\fig...
1,condition_overall_ORA_heatmap,G:\Repository\glioma-cnv-single-cell-atlas\fig...
2,program_score_global_barplot,G:\Repository\glioma-cnv-single-cell-atlas\fig...


Figure manifest shape: (3, 2)


In [12]:
# =========================
# 11. Output manifest and final checkpoints
# =========================
outputs = [
    METADATA_DIR
    / f"{OUT_PREFIX}_readiness.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_legacy_direction_detection_audit.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_detected_DE_columns.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_gene_universe_report.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_gene_set_inventory.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_external_gmt_files_loaded.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_directional_gene_list_QC.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_gene_list_inventory.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_malignant_state_marker_ORA.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_condition_overall_malignant_ORA.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_within_state_condition_ORA.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_biological_interpretation_summary.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_program_score_global_kruskal_summary.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_program_score_pairwise_summary.tsv",
    METADATA_DIR
    / f"{OUT_PREFIX}_figure_manifest.tsv",
]


manifest_df = pd.DataFrame(
    [
        {
            "path": str(
                path
            ),
            "exists": Path(
                path
            ).exists(),
            "size_mb": (
                round(
                    Path(
                        path
                    ).stat().st_size
                    / 1024**2,
                    3,
                )
                if Path(
                    path
                ).exists()
                else np.nan
            ),
        }
        for path
        in outputs
    ]
)


write_tsv_replace(
    manifest_df,
    METADATA_DIR
    / f"{OUT_PREFIX}_output_manifest.tsv",
)


if manifest_df.shape != (
    15,
    3,
):
    raise RuntimeError(
        f"Canonical enrichment manifest checkpoint failed: {manifest_df.shape}"
    )


if not manifest_df[
    "exists"
].all():
    raise RuntimeError(
        "One or more canonical Notebook-23 outputs are missing."
    )


display(
    manifest_df
)


print(
    "\nFINAL CHECKPOINTS"
)


print(
    "Historical notebook mapping:",
    "22_functional_enrichment_pathway_interpretation_HVG8000_r0_6.ipynb "
    "-> 23_curated_enrichment.ipynb",
)


print(
    "Scientific correction:",
    "direction-aware DE column detection",
)


print(
    "Historical direction-detection failure reproduced:",
    bool(
        legacy_missing_all.all()
    ),
)


print(
    "Corrected DE columns:",
    detected_columns[
        [
            "label",
            "lfc_col",
            "q_col",
            "p_col",
        ]
    ].to_dict(
        orient="records"
    ),
)


print(
    "Canonical input rows:",
    EXPECTED_INPUT_ROWS,
)


print(
    "Gene universe:",
    len(
        universe_genes
    ),
)


print(
    "Curated gene sets:",
    len(
        gene_sets
    ),
)


print(
    "External GMT enabled:",
    ENABLE_EXTERNAL_GMTS,
)


print(
    "Directional gene lists:",
    direction_qc.shape,
)


print(
    "Directional list counts:",
    direction_qc.groupby(
        [
            "analysis",
            "direction",
        ]
    )
    .size()
    .to_dict(),
)


print(
    "State ORA:",
    state_enrich.shape,
)


print(
    "Condition-overall ORA:",
    condition_overall_enrich.shape,
)


print(
    "Within-state ORA:",
    within_state_enrich.shape,
)


print(
    "Interpretation summary:",
    interpretation_summary.shape,
)


print(
    "Program global summary:",
    program_global_out.shape,
)


print(
    "Program pairwise summary:",
    program_pairwise_out.shape,
)


print(
    "Figure manifest:",
    figure_manifest.shape,
)


print(
    "Output manifest:",
    manifest_df.shape,
)


print(
    "\n[DONE] Notebook 23 corrected direction-aware curated enrichment completed."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\functional_enrichment_HVG8000_r0_6_output_manifest.tsv shape=(15, 3)


,path,exists,size_mb
0,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
1,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
2,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
3,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
4,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
5,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.012
7,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.006
8,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.010
9,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.013



FINAL CHECKPOINTS
Historical notebook mapping: 22_functional_enrichment_pathway_interpretation_HVG8000_r0_6.ipynb -> 23_curated_enrichment.ipynb
Scientific correction: direction-aware DE column detection
Historical direction-detection failure reproduced: True
Corrected DE columns: [{'label': 'malignant_state_significant', 'lfc_col': 'logFC_logCPM_state_vs_rest', 'q_col': 'padj_fdr_bh', 'p_col': 'pval_mannwhitney'}, {'label': 'malignant_state_top', 'lfc_col': 'logFC_logCPM_state_vs_rest', 'q_col': 'padj_fdr_bh', 'p_col': 'pval_mannwhitney'}, {'label': 'condition_overall_significant', 'lfc_col': 'log2FC_A_vs_B', 'q_col': 'padj_welch', 'p_col': 'pvalue_welch'}, {'label': 'condition_overall_top', 'lfc_col': 'log2FC_A_vs_B', 'q_col': 'padj_welch', 'p_col': 'pvalue_welch'}, {'label': 'within_state_condition_top', 'lfc_col': 'log2FC_A_vs_B', 'q_col': 'padj_welch', 'p_col': 'pvalue_welch'}]
Canonical input rows: {'state_significant': 20800, 'state_top': 1000, 'condition_overall_significant': 

## What changed relative to the historical executed notebook?

Only the DE-column interpretation layer is intentionally corrected.

The historical executed notebook itself proves the problem: its `detected_DE_columns` output showed `None` for effect size, FDR, and p-value in every DE table. The ORA engine therefore received undirected lists while labeling them as `up`.

This notebook recognizes the exact canonical upstream columns and verifies their signs before ORA.

The Fisher exact test, BH-FDR procedure, 15,176-gene universe, 16 curated gene sets, program-score summaries, and figure construction remain based on the historical analysis design.

Because this is a real scientific correction rather than a memory-only refactor, the corrected ORA row counts and pathway rankings are expected to differ from the historical 66-state / 67-condition rows. The final checkpoint deliberately reports those corrected values instead of forcing the old incorrect results.